<a href="https://colab.research.google.com/github/GoudoMahan/AI-agent-practice/blob/main/lab3_task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Lab 3 Task 1：Variational Autoencoders 变分自编码器


姓名：胡豪达

学号：523021910471

本 Task 占比本 Lab 评分的 50%，分为两个部分：

| 部分 | 内容 | 分值 | 任务 |
|------|------|------|------|
| Part 1 | 认识 VAE 与 latent space | 25 分 | 理解概念并运行所有的代码块 |
| Part 2 | 动手做一做：权衡保真度和连续性 | 75 分 | 填写表格体会 beta 变化对生成结果的影响 |

<mark> 👉  高亮块是你需要完成或者修改的内容提示，你需要根据任务要求修改对应地方下方的代码块 </mark>，其余代码块请按顺序运行

---
## 🔧 环境安装

In [ ]:
!pip install -q torchinfo datasets

In [ ]:
import datetime
import sys

from IPython.display import HTML, clear_output, display
from PIL import Image as pil_image
import ipywidgets as widgets
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits import mplot3d
import numpy as np
from scipy import stats
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from tqdm.notebook import tqdm
import torchinfo

%matplotlib inline

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed: int) -> None:
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

---
## Part 1. 什么是 VAE (变分自编码器)？—— 为现实世界建立“连续的”数学坐标系

![VAE 结构图](https://i.ibb.co/wbPsmrQ/vae.png)

变分自编码器（VAE）是一种编码器–解码器模型。模型先把输入通过编码器映射到潜空间，再从潜向量解码并重建输入。

VAE 对潜空间施加约束，使其接近单位高斯分布，因此可以通过从高斯分布采样得到合理的潜向量。整个潜空间被连续的分布填满且结构化，那么在空间中任意游走，或者在两个事物的特征之间做线性插值，解码器都能输出有意义且平滑过渡的结果。

VAE 的训练过程，本质上是寻找“保真度”与“空间规律性”之间的平衡。它的损失函数由两项相互制约的目标组成：

1. **重构误差 (Reconstruction Loss)**：负责“保真”。它要求模型解码出来的结果必须尽可能还原输入数据，确保在压缩和解压的过程中，事物的核心语义（如轮廓、类别）没有丢失。
2. **KL 散度 (KL Divergence)**：负责“规律”。它像是一个空间约束器，强迫所有数据的特征分布必须向标准正态分布靠拢。如果没有 KL 散度，模型就会为了追求极致的重构精度，退化回那个死记硬背的普通自编码器，导致潜空间再次出现断层。


单看生成的图像质量，VAE 往往有些模糊，并不如后来的 GAN 锐利。但 VAE 真正的历史贡献不在于直接生成最终产品，而在于它**提供了一种将高维现实世界降维、并梳理成结构化且可计算的“潜空间 (Latent Space)”的范式**。 如今大放异彩的 Stable Diffusion，其全称是 Latent Diffusion Model。如果没有 VAE 提前将巨大的像素网格压缩成语义密集的连续低维特征，目前的消费级显卡根本无法计算复杂的扩散过程。VAE 就像是 AI 世界的底层测绘员，把复杂的感官数据转化成了机器可以进行加减乘除的坐标系。

下面，我们在 MNIST 手写数字数据集上训练一个简单的 VAE 模型。

它有两个输入:
1. 给编码器用的图像 x
2. 给解码器用的采样 eps（来自 $\mathcal{N}(0, 1)$ 的随机噪声），解码时会通过重参数化技巧与 $\mu_x$、$\sigma_x$ 结合使用。

> 重参数化技巧：若用普通采样（直接用编码器输出的均值和标准差去采样），则无法端到端训练，因为采样不是可微的操作。要解决这个问题，可以利用正态分布的一个性质：从 $N(\mu_\psi, \sigma_\psi)$ 采样，等价于 $z = \mu_\psi + \sigma_\psi \odot \epsilon,\ \epsilon \sim N(0, I)$。这样一来，“采样”里对随机性的部分只是与编码器对 $\sigma_\phi$ 的预测相乘的一个因子，梯度就能从输出一路传回编码器。

你可以阅读其中的注释，了解他们的作用。

In [ ]:
# Negative log likelihood loss (Bernoulli)
def nll(y_true, y_pred, reduction='sum'):
    loss = F.binary_cross_entropy(y_pred, y_true, reduction='none').sum(dim=-1)

    if reduction == 'sum':
        return loss.sum()
    elif reduction == 'mean':
        return loss.mean()
    else:
        return loss

# KL Divergence
def kl(mu, log_var, reduction='sum'):
    loss = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp()).sum(dim=1)
    if reduction == 'sum':
        return loss.sum()
    elif reduction == 'mean':
        return loss.mean()
    else:
        return loss

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super(VAE, self).__init__()
        # Define latent dimension
        self.latent_dim = latent_dim

        # Encoder
        # 编码器 $\phi$ 的作用是用 $Q_\phi(z|x_i)$ 去逼近 $P_\theta(z|x_i)$
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=5, stride=2, padding=2),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, kernel_size=5, stride=2, padding=2),
            nn.LeakyReLU(0.2),
            nn.Flatten()
        )

        # 编码器要针对不同输入 $x$（即输入图像）估计参数 $\mu_x$、$\sigma_x$，也就是对每个潜变量维输出均值与标准差
        # Compute μ and log(σ²)
        self.z_mu = nn.Linear(128*7*7, latent_dim)
        self.z_log_var = nn.Linear(128*7*7, latent_dim)

        # Decoder
        # 解码器：以从 $\mathcal{N}(\mu_\psi, \sigma_\psi)$ 中采样得到的向量为输入，输出一张图像。
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128*7*7),
            nn.LeakyReLU(0.2),
            nn.Unflatten(1, (128, 7, 7)),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, kernel_size=5, padding=2),
            nn.LeakyReLU(0.2),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(64, 1, kernel_size=5, padding=2),
            nn.Sigmoid(),
            nn.Flatten()
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.z_mu(h), self.z_log_var(h)

    def reparameterize(self, mu, log_var):
        ##### Reparametrisation trick
        # 若用普通采样（直接用编码器输出的均值和标准差去采样），则无法端到端训练，因为采样不是可微的操作。
        # 重参数化（reparametrisation）技巧能够使梯度就能从输出一路传回编码器。

        # Log_var to sigma
        std = torch.exp(0.5 * log_var)
        # Define a separate noise input (which must be provided during training)
        eps = torch.randn_like(std)
        # Multiply sigma with the external noise and add mu to obtain the latent vector
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var

torchinfo.summary(VAE(latent_dim=2), input_size=(1, 1, 28, 28))

接下来训练模型若干轮，看能否较好地拟合数据。训练结束后，将使用训练好的 VAE 生成新样本。训练时间约需 3 分钟，你可以调大 `n_epoch` 以进行更长时间的训练来获得更好的模型性能。

In [ ]:
# Data
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transforms.ToTensor())

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

In [ ]:
set_seed(42)

latent_dim = 2
model = VAE(latent_dim).to(DEVICE)
optimizer = optim.Adam(model.parameters())

# Training the model
best_loss = float('inf')
patience = 3
no_improve_epochs = 0

n_epoch = 20

for epoch in range(1, n_epoch+1):
    model.train()
    train_loss = 0
    for batch_idx, (data, _) in enumerate(
        tqdm(train_loader, desc=f'Epoch {epoch}/{n_epoch}', leave=False)
    ):
        data = data.to(DEVICE)
        optimizer.zero_grad()

        recon_batch, mu, log_var = model(data)
        nll_loss = nll(data.view(-1, 784), recon_batch)
        kl_loss = kl(mu, log_var)

        loss = nll_loss + kl_loss
        loss.backward()
        train_loss += loss.item()
        optimizer.step()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(DEVICE)
            recon_batch, mu, log_var = model(data)
            nll_loss = nll(data.view(-1, 784), recon_batch)
            kl_loss = kl(mu, log_var)
            val_loss += (nll_loss + kl_loss).item()

    train_loss /= len(train_loader.dataset)
    val_loss /= len(test_loader.dataset)
    print(f'Epoch: {epoch}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

    if val_loss < best_loss:
        best_loss = val_loss
        no_improve_epochs = 0
        torch.save(model.state_dict(), 'best_vae_model.pth')
    else:
        no_improve_epochs += 1
        if no_improve_epochs >= patience:
            print(f'Early stopping after {epoch} epochs')
            model.load_state_dict(torch.load('best_vae_model.pth'))
            break

# Calculate MSE on test set
model.eval()
total_mse = 0
with torch.no_grad():
    for data, _ in test_loader:
        data = data.to(DEVICE)
        recon_batch, mu, _, = model(data)
        mse = F.mse_loss(recon_batch, data.view(-1, 784), reduction='sum')
        total_mse += mse.item()

print(f"Test MSE: {total_mse / len(test_loader.dataset):.4f}")

运行下方代码：我们可以把在测试集各图像的 $\mu_x$ 的分布绘制出来。不同的颜色代表不同类别的手写数字图片。我们会发现：

1. 编码后均值 $\mu$ 的分布也体现出按类别成簇的现象。
2. 由于损失里库尔贝克–莱布勒（KL）散度项的作用，这种分布还呈现出围绕中心大致呈环状/圆形的形态：训练会促使编码器输出的 $\mu_x$ 接近 0，而 $\sigma_x$ 接近 1。

In [ ]:
# Plot latent space distribution
model.eval()
latent_vectors = []
labels = []

with torch.no_grad():
    for data, target in test_loader:
        data = data.to(DEVICE)
        mu, _ = model.encode(data)
        latent_vectors.append(mu.cpu())
        labels.append(target.cpu())

z_test = torch.cat(latent_vectors, dim=0).numpy()
y_test = torch.cat(labels, dim=0).numpy()

# Display a 2D plot of the digit classes in the latent space
fig, ax = plt.subplots(figsize=(6,6))
cm = plt.get_cmap('gist_rainbow')
ax.set_prop_cycle(color=[cm(1.*i/(10)) for i in range(10)])
for l in range(10):
    # Only select indices for corresponding label
    ind = y_test == l
    ax.scatter(z_test[ind, 0], z_test[ind, 1], label=str(l), s=10)
ax.legend()
plt.title("Latent distribution")
plt.show()

接下来，利用训练好的模型，我们可以为潜变量做一个简洁的动画：展示用于生成数据时在 $z$ 中选取的点，以及与之对应生成出的图像。

In [ ]:
## We create a 2d array
# Number of points to use, increase it for smoother animation
# Using more points makes the function slower
n_points = 50
# theta from 0 to 2pi
theta = np.linspace(0, 2*np.pi, n_points)
# radius of the circle (change it depending on your reprentation space plot)
r = 1
# compute x and y (you can add an offset depending on your latent space)
offset_x = 0
offset_y = 0
x = r*np.cos(theta) + offset_x
y = r*np.sin(theta) + offset_y
latent = np.stack([x, y], -1)

## We now plot as before the 2d scatter with the images from the test set
## and the corresponding label
# Get test latent vectors
model.eval()
with torch.no_grad():
    z_test = []
    for data, _ in test_loader:
        mu, _ = model.encode(data.to(DEVICE))
        z_test.append(mu.cpu())
    z_test = torch.cat(z_test).numpy()

# Create figure
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

# First subplot: latent space with test points
ax[0].set_prop_cycle(color=[plt.cm.gist_rainbow(1.*i/10) for i in range(10)])
for l in range(10):
    ind = y_test == l
    ax[0].scatter(z_test[ind, 0], z_test[ind, 1], label=str(l), s=10)
ax[0].legend()
scat = ax[0].scatter(latent[0,0], latent[0,1], s=200, c='k')

# Second subplot: generated image
with torch.no_grad():
    latent_tensor = torch.FloatTensor(latent).to(DEVICE)
    latent_im = model.decoder(latent_tensor).cpu().numpy()

im = ax[1].imshow(latent_im[0].reshape(28, 28), cmap='gray')
ax[1].axis('off')

# Animation update function
def updatefig(i):
    scat.set_offsets(latent[i])
    im.set_array(latent_im[i].reshape(28, 28))
    return im,

# Create animation
anim = animation.FuncAnimation(fig, updatefig, frames=n_points, interval=200, blit=True)
plt.close()
HTML(anim.to_html5_video())

## Part 2: 🧪 动手做一做：寻找最优的“压缩压力” ($\beta$ 参数)

在上一节中，我们了解了 VAE 的损失函数是由两部分组成的：
1. **重构误差 (Reconstruction Loss)**：负责“死记硬背”。它要求生成的图像必须和原图一模一样（保真度）。
2. **KL 散度 (KL Divergence)**：负责“归纳整理”。它要求把提取到的特征按照标准正态分布的规律，整整齐齐地收纳在潜空间里（连续性）。

传统的 VAE 认为这两项任务一样重要，它们的权重比例是 1:1。但在 2017 年，研究人员提出了一个极其简单却绝妙的改进：**在 KL 散度前面加一个权重参数 $\beta$ (Beta)**。

$$\mathcal{L} = \text{重构误差} + \beta \times \text{KL 散度}$$

$\beta$ 是什么？它是“整理收纳的强制力”。你可以把训练 AI 想象成“把一堆形状各异的行李塞进一个有弹性的行李箱”：

* **当 $\beta = 1$ (传统 VAE)**：相当于箱子比较松垮。AI 会为了尽量不弄坏行李（降低重构误差），把衣服、鞋子、书本全乱七八糟地缠绕在一起塞进去。结果就是：**特征纠缠 (Entanglement)**。比如你只想让生成的人脸“笑一下”，结果它的头发变长了，肤色也变了，因为“笑容”和“长发”在箱子里缠在了一起。
* **当 $\beta > 1$ ($\beta$-VAE)**：相当于给箱子加上了极强的真空压缩压力。AI 面临巨大的“收纳压力”，被迫抛弃那些琐碎的细节，**只保留最核心的属性**（如单独的笑容、单独的脸型），并且把它们整整齐齐、互不干扰地分开放好。结果就是：**特征解耦 (Disentanglement)**。此时，你调整“笑容”的控制旋钮，就真的只有嘴角在动，其他部位纹丝不动。


在生成式模型中，我们面临一个永恒的权衡：生成图像的质量 (Quality) vs 特征的可解释性 (Disentanglement)。

在这个任务中你需要通过损失函数中的 beta 值来手动控制了对“潜空间分布”的约束强度：
* **低 $\beta$ (例如 1):** 图像清晰，但特征杂乱。改变一个维度会引起全身变化。
* **高 $\beta$ (例如 > 10):** 图像模糊，但特征清晰。改变一个维度只影响一个语义（如：只改变头发颜色）。




<mark> 👉  请你在下方代码块中修改 `beta` 参数，分别训练并运行模型并观察输出的 特征遍历图 (Latent Traversal)，观察改变单一维度时，图像是否发生了“语义污染”？ 请根据你的观察，填写下面的表格 </mark>。




| 实验组 | $\beta$ 取值 | 重构图像清晰度 (还原度) | 特征解耦程度 (是否只改变单一属性) | 你的发现 (用一句话描述结果) |
| :--- | :--- | :--- | :--- | :--- |
| **对照组** | $\beta = 1$ |  |  | |
| **实验组 A** | $\beta = $ |  |  | |
| **实验组 B** | $\beta = $ |  |  | |
| **实验组 C** | $\beta = $ |  |  | |
| **实验组 D** | $\beta = $ |  |  | |
| ... | $\beta = $ |  |  | |



本实现将在 CelebA 数据集上训练，CelebA 数据集包含大量名人的彩色人脸图像，并带有诸如「微笑」「戴眼镜」「涂口红」等属性标签，还提供边界框和人脸部位定位信息。CelebA 是常用数据集，广泛用于人脸属性识别、人脸检测、关键点（或人脸部位）定位，以及人脸编辑与合成。

本次作业使用的子集包含 10,000 张训练图像、1,000 张验证图像和 1,000 张测试图像。

In [ ]:
from pathlib import Path
from torch.utils.data import DataLoader, Dataset, Subset

import torchvision.datasets as tvd
from datasets import load_dataset

set_seed(42)
CELEBA_ROOT = './data/celeba'
CELEBA_SOURCE: str = 'huggingface'
IMG_SIZE = 64
N_TRAIN, N_VAL, N_TEST = 10_000, 1_000, 1_000

img_tf = transforms.Compose(
    [transforms.Resize(IMG_SIZE), transforms.CenterCrop(IMG_SIZE), transforms.ToTensor()]
)


class _CelebAFromHuggingFace(Dataset):
    """CelebA 训练集, 与 torchvision 相同规模 (约 162k), 不经过 gdown."""

    def __init__(self) -> None:
        print('Loading train split from Hugging Face: ryushinn/CelebA (first time ~1.4GB download) ...')
        self._ds = load_dataset('ryushinn/CelebA', split='train')
        self._tf = img_tf

    def __len__(self) -> int:
        return len(self._ds)

    def __getitem__(self, index: int):
        img = self._ds[index]['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        t = self._tf(img)
        return t, 0


def _torchvision_cache_ready() -> bool:
    p = Path(CELEBA_ROOT) / 'celeba' / 'img_align_celeba'
    if not p.is_dir():
        return False
    return any(p.iterdir())


if CELEBA_SOURCE == 'huggingface':
    train_full: Dataset = _CelebAFromHuggingFace()
elif CELEBA_SOURCE == 'torchvision':
    dload = not _torchvision_cache_ready()
    if dload:
        print('Downloading (or loading) CelebA via torchvision (GDrive) ... 若失败请改 CELEBA_SOURCE=\'huggingface\'')
    train_full = tvd.CelebA(CELEBA_ROOT, split='train', target_type='attr', download=dload, transform=img_tf)
else:
    raise ValueError("CELEBA_SOURCE must be 'huggingface' or 'torchvision'")
n_all = N_TRAIN + N_VAL + N_TEST
g = torch.Generator().manual_seed(0)
order = torch.randperm(len(train_full), generator=g)[:n_all].tolist()

sub_train = Subset(train_full, order[:N_TRAIN])
sub_val = Subset(train_full, order[N_TRAIN : N_TRAIN + N_VAL])
sub_test = Subset(train_full, order[N_TRAIN + N_VAL :])

# Match Part 1 batching (CPU-friendly; raise if you use GPU and want more throughput)
BATCH = 32
num_workers = 0

celeba_train_loader = DataLoader(
    sub_train, batch_size=BATCH, shuffle=True, num_workers=num_workers, pin_memory=False
)
celeba_val_loader = DataLoader(
    sub_val, batch_size=BATCH, shuffle=False, num_workers=num_workers, pin_memory=False
)
celeba_test_loader = DataLoader(
    sub_test, batch_size=BATCH, shuffle=False, num_workers=num_workers, pin_memory=False
)

# —— 可视化: 2x3 网格, 高分辨率, 去坐标轴, 统一样式 ——
try:
    plt.style.use('seaborn-v0_8-white')
except OSError:
    try:
        plt.style.use('seaborn-white')
    except OSError:
        pass

n_show = 6
x_batch, _ = next(iter(celeba_train_loader))
n_show = min(n_show, x_batch.shape[0])
fig, axes = plt.subplots(2, 3, figsize=(7.2, 5.0), dpi=150)
fig.patch.set_facecolor('white')
fig.suptitle('CelebA (subset) — 64×64, RGB, [0,1]', fontsize=12, y=1.01)
for i, ax in enumerate(axes.flat):
    if i < n_show:
        im = x_batch[i].cpu().permute(1, 2, 0).numpy()
        im = np.clip(im, 0.0, 1.0)
        ax.imshow(im, interpolation='bilinear', aspect='equal')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(True)
    for s in ax.spines.values():
        s.set_color('#d0d0d0')
        s.set_linewidth(0.6)
plt.tight_layout()
plt.show()
print(
    'Train/Val/Test counts:',
    N_TRAIN,
    N_VAL,
    N_TEST,
    '| loader target:',
    ("HuggingFace ryushinn/CelebA (train)" if CELEBA_SOURCE == 'huggingface' else str(Path(CELEBA_ROOT).resolve())),
)

<mark> 👉  修改下面的 beta 多次运行下方的代码块，观测现象并填写上方的表格 </mark>

训练完成会输出三张图：

1. 编码器从真实人脸压缩到隐变量，再经解码器还原出像素：观察原图和重建是否像。beta 小往往更清晰、细节多; beta 大时 KL 更强，常会更糊、细节少，但潜空间更规整。

2. 不从数据编码，而是直接从标准高斯里采 z,只过解码器得到人脸：模型学到的在训练分布上合理的生成，相当于无条件的随机造脸。VAE 训练时要求后验接近先验，所以这一格能看先验里的样本在解码后像不像人；beta 大时，这类样本往往更糊、更平均，但有时更稳定、不过拟合到奇怪纹理。

3. 在 z=0 附近，只动某一个隐变量维，看解码图像沿这一维连续怎么变: 只变一种东西（例如只变笑、或只变朝向）/ 很多属性一起变（发型、光、脸同时动）

In [ ]:
beta: float = 1.0           # <---- 请修改这里，分别尝试不同的 $\beta$ 值
LATENT_DIM: int = 32
C_H, C_W = 16, 16
N_FLAT = 128 * C_H * C_W


class CelebAVAE(nn.Module):
    def __init__(self, latent_dim: int) -> None:
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5, stride=2, padding=2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=5, stride=2, padding=2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
        )
        self.z_mu = nn.Linear(N_FLAT, latent_dim)
        self.z_log_var = nn.Linear(N_FLAT, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, N_FLAT),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Unflatten(1, (128, C_H, C_W)),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(128, 64, kernel_size=5, padding=2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(64, 3, kernel_size=5, padding=2),
            nn.Sigmoid(),
            nn.Flatten(),
        )

    def encode(self, x: torch.Tensor) -> tuple:
        h = self.encoder(x)
        return self.z_mu(h), self.z_log_var(h)

    def reparameterize(self, mu: torch.Tensor, log_var: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)

    def forward(self, x: torch.Tensor) -> tuple:
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var


def bce_nll_batch_sum(y_true: torch.Tensor, y_pred: torch.Tensor) -> torch.Tensor:
    b = y_true.size(0)
    return F.binary_cross_entropy(y_pred, y_true.view(b, -1), reduction='sum')

set_seed(42)
celeba_vae = CelebAVAE(LATENT_DIM).to(DEVICE)
opt = optim.Adam(celeba_vae.parameters(), lr=1e-3)
torchinfo.summary(celeba_vae, input_size=(1, 3, IMG_SIZE, IMG_SIZE), device=DEVICE)

best_v = float('inf')
no_improve = 0
pat = 4
epochs = 25
ckpt = f'best_celeba_b{beta}_z{LATENT_DIM}.pth'


def _load_state(p: str):
    try:
        return torch.load(p, map_location=DEVICE, weights_only=True)
    except TypeError:
        return torch.load(p, map_location=DEVICE)


print('Training (beta, latent_dim) =', beta, LATENT_DIM)
for ep in range(1, epochs + 1):
    celeba_vae.train()
    tr = 0.0
    for xb, _ in tqdm(celeba_train_loader, desc=f'ep{ep}', leave=False):
        xb = xb.to(DEVICE)
        opt.zero_grad()
        rec, mu, log_var = celeba_vae(xb)
        nll_b = bce_nll_batch_sum(xb, rec)
        kl_b = kl(mu, log_var, reduction='sum')
        loss = nll_b + float(beta) * kl_b
        loss.backward()
        tr += loss.item()
        opt.step()
    tr /= N_TRAIN

    celeba_vae.eval()
    vsum = 0.0
    with torch.no_grad():
        for xb, _ in celeba_val_loader:
            xb = xb.to(DEVICE)
            rec, mu, log_var = celeba_vae(xb)
            vsum += bce_nll_batch_sum(xb, rec) + float(beta) * kl(mu, log_var, reduction='sum')
    vsum /= N_VAL

    print(
        f'Epoch {ep:2d} | train loss/img (NLL+β·KL, sum norm like Part1): {tr:8.1f} | val /img: {vsum:8.1f}'
    )
    if vsum < best_v + 1e-9:
        best_v, no_improve = vsum, 0
        torch.save(celeba_vae.state_dict(), ckpt)
    else:
        no_improve += 1
        if no_improve >= pat:
            print('Early stop; loading best', ckpt)
            celeba_vae.load_state_dict(_load_state(ckpt))
            break
else:
    celeba_vae.load_state_dict(_load_state(ckpt))

# ---- inference visualizations: reconstructions + samples + latent traversal ----
celeba_vae.eval()
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    try:
        plt.style.use('seaborn-whitegrid')
    except OSError:
        pass
fig_kw = dict(dpi=140, facecolor='white')


def to_rgb_hw(t: torch.Tensor) -> np.ndarray:
    t = t.detach().cpu()
    if t.dim() == 1:
        t = t.view(3, IMG_SIZE, IMG_SIZE)
    return np.clip(t.permute(1, 2, 0).numpy(), 0, 1)


# (1) Original vs reconstruction — 2×8
with torch.no_grad():
    xb, _ = next(iter(celeba_test_loader))
    xb = xb.to(DEVICE)
    nvis = 8
    if xb.size(0) < nvis:
        nvis = xb.size(0)
    xb = xb[:nvis]
    rec, _, _ = celeba_vae(xb)
    rec_ = rec.view(-1, 3, IMG_SIZE, IMG_SIZE)

fig, axs = plt.subplots(2, nvis, figsize=(1.35 * nvis, 2.6), **fig_kw)
fig.suptitle('Reconstruction: top = input, bottom = VAE output (beta = %.2g)' % (beta,), fontsize=12, y=0.99)
for j in range(nvis):
    axs[0, j].imshow(to_rgb_hw(xb[j]), interpolation='bilinear')
    axs[0, j].set_axis_off()
    axs[1, j].imshow(to_rgb_hw(rec_[j]), interpolation='bilinear')
    axs[1, j].set_axis_off()
plt.tight_layout()
plt.show()

# (2) Prior samples z ~ N(0, I)
n_row, n_col = 3, 4
z_samp = torch.randn(n_row * n_col, LATENT_DIM, device=DEVICE)
with torch.no_grad():
    gen = celeba_vae.decode(z_samp).view(-1, 3, IMG_SIZE, IMG_SIZE).cpu()
fig, axs = plt.subplots(n_row, n_col, figsize=(1.1 * n_col, 1.0 * n_row + 0.2), **fig_kw)
fig.suptitle('Prior: z ~ N(0, I) then decode (beta = %.2g)' % (beta,), fontsize=12)
for i, ax in enumerate(axs.flat):
    ax.imshow(to_rgb_hw(gen[i]), interpolation='bilinear')
    ax.set_axis_off()
plt.tight_layout()
plt.show()

# (3) Latent traversal — first 6 dimensions, 11 steps in [-2.5, 2.5] (classical for β-VAE sweeps)
n_d = min(6, LATENT_DIM)
n_steps = 11
t_vals = torch.linspace(-2.5, 2.5, n_steps)
fig, axs = plt.subplots(n_d, n_steps, figsize=(0.7 * n_steps, 0.7 * n_d + 0.2), **fig_kw)
fig.suptitle('Latent traversal: vary one z at a time (other dims = 0) | beta = ' + str(beta), fontsize=12, y=1.02)


def _ax_traverse(d: int, s: int):
    if n_d == 1:
        return axs[s]
    return axs[d, s]


with torch.no_grad():
    for d in range(n_d):
        for s in range(n_steps):
            z = torch.zeros(1, LATENT_DIM, device=DEVICE)
            z[0, d] = t_vals[s]
            o = celeba_vae.decode(z).view(1, 3, IMG_SIZE, IMG_SIZE)
            axd = _ax_traverse(d, s)
            axd.imshow(to_rgb_hw(o[0]), interpolation='bilinear')
            axd.set_axis_off()
            if s == 0:
                axd.set_ylabel(f'z_{d}', fontsize=8, rotation=0, labelpad=12, va='center')
        if d == 0:
            for s in range(n_steps):
                _ax_traverse(0, s).set_title(f'{t_vals[s].item():.1f}', fontsize=7, pad=2)
plt.tight_layout()
plt.show()
print('Done. Checkpoint:', ckpt, '| val loss (best):', best_v)

<mark> 👉 既然 $\beta$ 越大，AI 对世界的理解就越“条理清晰”（特征解耦），那我们为什么不把 $\beta$ 设为 100 甚至更高呢？请结合你观察到的图像清晰度给出理由, 写在下方的 markdown cell 中。 </mark>